In [1]:
from pathlib import Path
import numpy as np
import rasterio
import pandas as pd
from tqdm import tqdm

RAW_DIR = Path("../data/raw")

BANDS = ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7", "ST_B10"]

In [2]:
def read_band(city_dir, band):
    path = list(city_dir.glob(f"*.{band}.tif"))[0]
    with rasterio.open(path) as src:
        return src.read(1).astype(np.float32)

records = []

for city_dir in tqdm(sorted(RAW_DIR.iterdir())):
    if not city_dir.is_dir():
        continue

    city = city_dir.name

    for band in BANDS:
        arr = read_band(city_dir, band)

        records.append({
            "city": city,
            "band": band,
            "shape": arr.shape,
            "min": float(np.nanmin(arr)),
            "max": float(np.nanmax(arr)),
            "mean": float(np.nanmean(arr)),
            "std": float(np.nanstd(arr)),
            "p2": float(np.percentile(arr, 2)),
            "p98": float(np.percentile(arr, 98)),
        })

df = pd.DataFrame(records)
df.head()

  0%|          | 0/23 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_40164\2952362944.py:4: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  return src.read(1).astype(np.float32)
100%|██████████| 23/23 [00:06<00:00,  3.42it/s]


,city,band,shape,min,max,mean,std,p2,p98
0,agra_001,SR_B2,"(558, 558)",0.027095,0.239601,0.078857,0.017502,0.038342,0.110777
1,agra_001,SR_B3,"(558, 558)",0.052670,0.292896,0.118258,0.020885,0.065072,0.155159
2,agra_001,SR_B4,"(558, 558)",0.036033,0.332331,0.130048,0.030423,0.055227,0.184515
3,agra_001,SR_B5,"(558, 558)",0.056534,0.398070,0.245112,0.036788,0.163921,0.304845
4,agra_001,SR_B6,"(558, 558)",0.032733,0.541840,0.233040,0.048970,0.135033,0.329595


In [5]:
df.groupby("band")[["min", "max", "mean", "std", "p2", "p98"]].mean()

,min,max,mean,std,p2,p98
band,,,,,,
SR_B2,0.021109,0.265465,0.071733,0.019065,0.037188,0.112615
SR_B3,0.043587,0.331359,0.107285,0.023439,0.065019,0.157316
SR_B4,0.031582,0.379039,0.117178,0.032080,0.056978,0.184237
SR_B5,0.044966,0.470180,0.224837,0.042729,0.134670,0.302381
SR_B6,0.026596,0.544729,0.209578,0.050525,0.106290,0.315106
SR_B7,0.017836,0.555623,0.162593,0.049175,0.066492,0.267020
ST_B10,300.056793,319.367193,308.935137,2.656366,303.757417,313.785802
